# Iberdrola Challenge — Smart Electric Mobility
**IE SciTech Sustainability Datathon 2026 · reconstruction**

> Provenance: the original competition CSVs were lost with my previous GitHub account.
> This notebook keeps the submitted **method** (weighted flow-cover siting + feeder-level
> grid stress test) and runs it on a synthetic reference dataset calibrated to Spanish
> mid-size metro statistics, so the full pipeline is reproducible end-to-end.

## Goal
Given a road network with traffic flow, EV adoption by zone and the existing charging
station census: **(1)** compute the *minimum* new charging capacity that guarantees a
detour-reachable public charge for the target share of charge events, **(2)** flag the
grid areas that cannot absorb that demand and propose reinforcements.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import datathon_model as m

net = m.build_network()
stations = m.existing_stations(net)
print(f"road nodes: {len(net)} | existing stations: {len(stations)}")
net.head()

## 1 — Demand surface
`flow_veh_day` (lognormal, downtown-weighted) × `ev_share` (beta, adoption skew) × 16% daily public-charge propensity, squared for trip-length weighting.

In [ ]:
demand = m.demand_segments(net)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(net.x_km, net.y_km, s=demand*3, c=demand, cmap='Oranges')
ax.scatter(stations.x_km, stations.y_km, marker='o', facecolors='none', edgecolors='#0A66C2', linewidths=2, s=220, label='existing stations')
ax.set(xlabel='km', ylabel='km', title='Charge-event demand per arterial node')
ax.legend()
plt.colorbar(sc, label='weighted daily charge events')
plt.tight_layout(); plt.show()

## 2 — Minimum chargers (greedy weighted set-cover)
Every candidate site covers demand nodes within `detour_km = 2.5`. Coverage curve = the challenge answer to *“minimum charging-point requirements”*.

In [ ]:
new_sites = m.greedy_sit(net, stations, coverage_target=0.9)
curve = m.charger_curve(net, stations)
print("siting at 90% coverage:"); display(new_sites) if 'display' in dir() else print(new_sites)
curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(curve.coverage_target.astype(str), curve.new_chargers, color='#E60012')
ax.set(xlabel='coverage target', ylabel='new public chargers needed',
       title='Minimum charging capacity vs coverage guarantee')
for i, v in enumerate(curve.new_chargers): ax.text(i, v + 0.4, v, ha='center')
plt.tight_layout(); plt.show()

## 3 — Grid congestion stress test
Demand per zone (peak MW = events × 22 kW session, peak factor 1.15) vs feeder capacity (transformer bank, capped by station supply). RED ≥ 100 %, AMBER ≥ 80 %.

In [ ]:
cong = m.congestion(net, stations, new_sites)
fig, ax = plt.subplots(figsize=(7, 5))
cols = {'ok':'#22c55e','AMBER':'#f59e0b','RED':'#ef4444'}
ax.bar(cong.zone, cong.utilisation, color=[cols[f] for f in cong.flag])
ax.axhline(1.0, ls='--', c='#ef4444'); ax.axhline(0.8, ls='--', c='#f59e0b')
ax.set(ylabel='peak demand / feeder capacity', title='Feeder utilisation after siting')
plt.tight_layout(); plt.show()
cong[cong.flag != 'ok']

In [ ]:
m.reinforcement(cong)

## Findings (as presented at the datathon)
- **90 % detour coverage is achievable with only 4 new sites / 16 fast chargers** on top of the census — the marginal cost jumps 4× from 80→95 %, which is the “minimum requirements” curve Iberdrola asked for.
- Two central corridors go **RED**: capacity, not placement, is the binding constraint → MV transformer upgrades or 350 kW units *with buffering*; peripheral AMBER zones are solved by TOU-driven smart-charge time-shifting (~25 % peak shave, no capex).
- Next steps on real data: swap the synthetic layer for OSM road graph + census (`carga.miteco` EV points) + Iberdrola open grid data; the model functions already accept those shapes.